In [2]:
## 데이터 정렬

import pandas as pd
import glob

# 1. 정렬 기준 순서 정의 (요청하신 순서대로)
custom_order = [
    '서울', '경기', '인천', '강원', '충남', '대전', '충북', '세종', 
    '경북', '대구', '경남', '부산', '울산', '전북', '전남', '광주', '제주'
]

# 2. [지역 컬럼용] 긴 행정구역 명칭을 짧은 명칭으로 바꾸는 사전
short_mapping = {
    '충청남도': '충남', '충청북도': '충북', '경상남도': '경남', '경상북도': '경북',
    '전라남도': '전남', '전라북도': '전북', '전북특별자치도': '전북',
    '제주특별자치도': '제주', '강원특별자치도': '강원', '세종특별자치시': '세종',
    '서울특별시': '서울', '부산광역시': '부산', '대구광역시': '대구',
    '인천광역시': '인천', '광주광역시': '광주', '대전광역시': '대전', '울산광역시': '울산'
}

# 3. [주소 컬럼용] 짧은 명칭을 다시 정식 명칭으로 바꾸는 사전
full_mapping = {
    '서울': '서울특별시', '경기': '경기도', '인천': '인천광역시', '강원': '강원특별자치도',
    '충북': '충청북도', '충남': '충청남도', '대전': '대전광역시', '세종': '세종특별자치시',
    '전북': '전북특별자치도', '전남': '전라남도', '광주': '광주광역시', '경북': '경상북도',
    '경남': '경상남도', '대구': '대구광역시', '울산': '울산광역시', '부산': '부산광역시', '제주': '제주특별자치도'
}

# --- 처리 함수 정의 ---

# 주소에서 '지역' 값을 정제해서 가져오는 함수
def get_clean_region(address):
    if not isinstance(address, str): return ""
    first_word = address.split()[0] # 첫 어절 추출
    
    # 사전에 있으면 짧은 이름 반환
    if first_word in short_mapping:
        return short_mapping[first_word]
    
    # '충청', '경상', '전라'로 시작하는 모호한 경우 주소 전체에서 '북'/'남' 확인
    if first_word.startswith('충청'): return '충북' if '북' in address else '충남'
    if first_word.startswith('경상'): return '경북' if '북' in address else '경남'
    if first_word.startswith('전라'): return '전북' if '북' in address else '전남'
    
    return first_word[:2] # 그 외엔 앞 두 글자

# 주소의 첫 어절을 정식 명칭으로 확장하는 함수
def expand_address(address):
    if not isinstance(address, str): return address
    parts = address.split()
    if not parts: return address
    
    # 첫 단어가 사전에 있으면 정식 명칭으로 교체
    if parts[0] in full_mapping:
        parts[0] = full_mapping[parts[0]]
    
    return " ".join(parts)

# --- 메인 실행 로직 ---

# 현재 폴더에서 'coffee_'로 시작하는 모든 .csv 파일을 찾음
csv_files = glob.glob('coffee_*.csv')

for file in csv_files:
    # 1. CSV 파일 읽기
    df = pd.read_csv(file)
    
    # '주소' 컬럼이 없는 파일은 건너뛰기
    if '주소' not in df.columns:
        print(f"[{file}] '주소' 컬럼이 없어 작업을 건너뜁니다.")
        continue
        
    print(f"[{file}] 처리 중...")
    
    # 작업 1: '지역' 컬럼 생성 및 보정
    region_values = df['주소'].apply(get_clean_region)
    if '지역' in df.columns:
        df['지역'] = region_values # 이미 있으면 덮어쓰기
    else:
        df.insert(0, '지역', region_values) # 없으면 맨 앞에 삽입
    
    # 작업 2: '주소' 컬럼 첫 어절 정식 명칭 변환
    df['주소'] = df['주소'].apply(expand_address)
    
    # 작업 3: 지정된 순서대로 정렬
    # '지역' 컬럼을 정해진 순서(custom_order)대로 정렬 가능한 범주형으로 변환
    df['지역'] = pd.Categorical(df['지역'], categories=custom_order, ordered=True)
    df = df.sort_values(by='지역')
    
    # 결과 저장 (이름 그대로 덮어쓰기)
    df.to_csv(file, index=False, encoding='utf-8-sig')
    print(f"[{file}] 작업 완료 및 저장.")

print("\n모든 대상 파일의 처리가 완료되었습니다.")

[coffee_공차.csv] 처리 중...
[coffee_공차.csv] 작업 완료 및 저장.
[coffee_더벤티.csv] 처리 중...
[coffee_더벤티.csv] 작업 완료 및 저장.
[coffee_던킨도너츠.csv] 처리 중...
[coffee_던킨도너츠.csv] 작업 완료 및 저장.
[coffee_디저트39.csv] 처리 중...
[coffee_디저트39.csv] 작업 완료 및 저장.
[coffee_메가커피.csv] 처리 중...
[coffee_메가커피.csv] 작업 완료 및 저장.
[coffee_메머드커피.csv] 처리 중...
[coffee_메머드커피.csv] 작업 완료 및 저장.
[coffee_빽다방.csv] 처리 중...
[coffee_빽다방.csv] 작업 완료 및 저장.
[coffee_스타벅스.csv] 처리 중...
[coffee_스타벅스.csv] 작업 완료 및 저장.
[coffee_이디야.csv] 처리 중...
[coffee_이디야.csv] 작업 완료 및 저장.
[coffee_컴포즈커피.csv] 처리 중...
[coffee_컴포즈커피.csv] 작업 완료 및 저장.
[coffee_투썸플레이스.csv] 처리 중...
[coffee_투썸플레이스.csv] 작업 완료 및 저장.
[coffee_파스쿠찌.csv] 처리 중...
[coffee_파스쿠찌.csv] 작업 완료 및 저장.
[coffee_폴바셋.csv] 처리 중...
[coffee_폴바셋.csv] 작업 완료 및 저장.
[coffee_하삼동커피.csv] 처리 중...
[coffee_하삼동커피.csv] 작업 완료 및 저장.
[coffee_할리스.csv] 처리 중...
[coffee_할리스.csv] 작업 완료 및 저장.

모든 대상 파일의 처리가 완료되었습니다.


In [ ]:
## 컬럼 추가 및 데이터 삽입

import pandas as pd

# 1. 작업할 파일명과 해당 파일에 넣을 브랜드 이름을 설정하세요.
file_name = 'coffee_던킨도너츠.csv'  # 수정 필요
brand_name = '던킨도너츠'             # 수정 필요

# 2. CSV 파일 불러오기
df = pd.read_csv(file_name)

# 3. '브랜드명' 컬럼 처리
# 이미 컬럼이 있다면 값을 업데이트하고, 없다면 맨 앞(0번 인덱스)에 삽입합니다.
if '브랜드명' in df.columns:
    df['브랜드명'] = brand_name
else:
    df.insert(0, '브랜드명', brand_name)

# 4. 결과 저장 (원래 파일 이름 그대로 덮어쓰기)
df.to_csv(file_name, index=False, encoding='utf-8-sig')

print(f"[{file_name}] 파일의 맨 앞에 '{brand_name}' 컬럼이 추가되었습니다.")

[coffee_던킨도너츠.csv] 파일의 맨 앞에 '던킨도너츠' 컬럼이 추가되었습니다.
